# DATA Preparation


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "32"  # To change accordingly tp yur threads

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from utils.load_data import load_target_dataset
from utils.clinical_data_features import ClinicalDataFeatures
from utils.molecular_data_features import MolecularDataFeature

## Target Dataset

In [ ]:
target_df = load_target_dataset("data/target_train.csv")

## The Clinical Dataset

In [ ]:
# Clinical Data

df_cli = pd.read_csv("data/X_train/clinical_train.csv")
df_cli_eval = pd.read_csv("data/X_test/clinical_test.csv")

# Apply the process
cdf = ClinicalDataFeatures()

df_cli = cdf.process(df_cli)
df_cli_eval = cdf.process(df_cli_eval)

## The molecular dataset

In [ ]:
df_mol = pd.read_csv("data/X_train/molecular_train.csv")
df_mol_eval = pd.read_csv("data/X_test/molecular_test.csv")
df_mol= df_mol[df_mol['ID'].isin(target_df['ID'])]

mdf = MolecularDataFeature()

df_mol = mdf.process(df_mol)
df_eval_mol = mdf.process(df_mol_eval )

## Merging & Aligning

In [ ]:
#We merge the processed molecular data with the clinical data
df = df_cli.merge(df_mol, on='ID', how='left')
#We fill NaN values with 0 for the new features
df = df.fillna(df.median(numeric_only=True))


df_eval = df_cli_eval.merge(df_eval_mol, on='ID', how='left')
#We fill NaN values with 0 for the new features
df_eval =df_eval.fillna(df.median(numeric_only=True))
#df.columns.to_series().to_csv("data/features.csv", index=False)

In [ ]:
#We allign the rows of the df with the target_df
df = df[df['ID'].isin(target_df['ID'])]
#We drop the ID column as it is not needed anymore
df.drop(columns=['ID', 'CENTER'], inplace=True)
df_eval.drop(columns=['ID', 'CENTER'], inplace=True)

target_df = target_df.drop(columns=['ID'])
target_df["OS_STATUS"]=target_df["OS_STATUS"].astype(bool)

We split for machine Learning

In [ ]:
# Now split
X_train, X_test, y_train, y_test = train_test_split(df, target_df, test_size=0.3, random_state=42)

In [ ]:
from sksurv.util import Surv
y_train_struct = Surv.from_dataframe(time = 'OS_YEARS', event= 'OS_STATUS', data = y_train)
y_test_struct = Surv.from_dataframe(time = 'OS_YEARS', event= 'OS_STATUS', data = y_test)

# Machine Learning

### A simple model

### More complex models

#### Gradient Boost

In [ ]:
from tqdm import tqdm
import random
from scipy.stats import uniform
import numpy as np
from sklearn.model_selection import KFold
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import gc
from sksurv.metrics import concordance_index_ipcw 
import json
# Set seeds for reproducibility
np.random.seed(67)
random.seed(67)

# Number of iterations and splits
n_iter = 200         # reduced from 100 to limit runtime & memory
n_splits = 5
n_jobs = 1

best_score = float('-inf')
best_params = None

def eval_one_fold(train_index, val_index, params, X_train, y_train_struct):
    try:
        X_train_fold = X_train.iloc[train_index]
        X_val_fold = X_train.iloc[val_index]
        y_train_fold = y_train_struct[train_index]
        y_val_fold = y_train_struct[val_index]

        # keep model lightweight where possible
        model = GradientBoostingSurvivalAnalysis(**params, random_state=42)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        score = concordance_index_ipcw(
            survival_train=y_train_fold,
            survival_test=y_val_fold,
            estimate=y_pred
        )[0]

        # explicitly delete heavy objects
        del model, X_train_fold, X_val_fold, y_train_fold, y_val_fold, y_pred
        gc.collect()
        return score
    except Exception as e:
        print(f"Fold failed with exception: {e}")
        return None

for i in tqdm(range(n_iter), desc="Random Search Progress"):
    # Random hyperparameters (narrower ranges to reduce extreme configs)
    params = {
        "learning_rate": float(uniform.rvs(loc=0.001, scale=0.05, random_state=i+42)),
        "n_estimators": int(uniform.rvs(loc=400, scale=2600, random_state=i+43)),
        "max_depth": int(uniform.rvs(loc=3, scale=15, random_state=i+44)),
        "min_samples_split": int(uniform.rvs(loc=5, scale=50, random_state=i+45)),
        "min_samples_leaf": int(uniform.rvs(loc=1, scale=12, random_state=i+46)),
        "subsample": float(uniform.rvs(loc=0.1, scale=0.7, random_state=i+47)),
        "dropout_rate": float(uniform.rvs(loc=0.0, scale=0.3, random_state=i+48)),
        "validation_fraction": 0.15
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42 + i)
    fold_scores = []
    # iterate folds sequentially to avoid parallel memory blowup
    for train_idx, val_idx in kf.split(X_train, y_train_struct):
        score = eval_one_fold(train_idx, val_idx, params, X_train, y_train_struct)
        if score is not None:
            fold_scores.append(score)

    if fold_scores:
        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            # write current best params to an outer file
            try:
                with open("best_param_GB.txt", "w") as fh:
                    json.dump(best_params, fh, default=lambda o: o.item() if hasattr(o, "item") else o, indent=2)
            except Exception:
                with open("best_param_GB.txt", "w") as fh:
                    fh.write(str(best_params))

    print(f"Iteration {i+1}: Mean score: {np.round(np.mean(fold_scores), 4) if fold_scores else 'nan'}", flush=True)
    # small cleanup every iteration
    gc.collect()

print("\nBest parameters:", best_params)
print("Best cross-validation score:", best_score)

The best model is : 




In [ ]:
# Load saved best parameters (JSON if available, fallback to Python literal)
with open("best_param_GB.txt", "r", encoding="utf-8") as fh:
    try:
        best_params = json.load(fh)
    except Exception:
        from ast import literal_eval
        fh.seek(0)
        best_params = literal_eval(fh.read())

# Fit the model with the loaded parameters
best_model_CGB = GradientBoostingSurvivalAnalysis(**best_params, random_state=42)  # type: ignore
best_model_CGB.fit(X_train, y_train_struct)

y_pred_cgboost = best_model_CGB.predict(X_test)
ipcw_c_index_cgboost = concordance_index_ipcw(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=y_pred_cgboost,
)

print(f"CGBoost IPCW Concordance Index on test: {ipcw_c_index_cgboost[0]}")

In [ ]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)
df_eval.fillna(0, inplace= True)

#We predict the survival function for the test set
y_pred_test = best_model_CGB.predict(df_eval)

In [ ]:
#We export the predictions to a csv file
df = pd.read_csv("data/X_test/clinical_test.csv")

predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_Gboost.csv', index=False)

#### Survival Forest

In [58]:
from sksurv.ensemble import RandomSurvivalForest
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_ipcw
from scipy.stats import uniform, randint
import numpy as np
import gc
import json

# safer config to avoid memory overflow
n_iter = 200              # reduce iterations
n_splits = 5             # CV folds
n_jobs = 1               # avoid parallel processes during CV

best_params = None
best_score = float("-inf")

def eval_one_fold(train_idx, val_idx, params, X_train, y_train_struct):
    try:
        X_train_fold = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        y_train_fold = y_train_struct[train_idx]
        y_val_fold = y_train_struct[val_idx]

        # keep model single-threaded to reduce memory peaks
        model = RandomSurvivalForest(**params, random_state=0, n_jobs=1)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        score = concordance_index_ipcw(
            survival_train=y_train_fold,
            survival_test=y_val_fold,
            estimate=y_pred
        )[0]

        # cleanup
        del model, X_train_fold, X_val_fold, y_train_fold, y_val_fold, y_pred
        gc.collect()
        return score
    except Exception as e:
        print(f"Fold failed with exception: {e}")
        return None

# random search (sequential fold evaluation to save memory)
rng = np.random.default_rng(42)
for i in tqdm(range(n_iter), desc="Random Search Progress"):
    params = {
        "n_estimators": int(rng.integers(100, 1001)),
        "max_depth": int(rng.integers(3, 21)) if rng.random() > 0.15 else None,
        "min_samples_split": int(rng.integers(2, 51)),
        "min_samples_leaf": int(rng.integers(1, 21)),
        "max_features": rng.choice(['sqrt', 'log2', None, 0.5, 0.7]),
        "min_weight_fraction_leaf": float(uniform.rvs(loc=0.0, scale=0.05, random_state=None)),
        # do not set n_jobs here (we pass to estimator in eval_one_fold)
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42 + i)
    fold_scores = []
    # evaluate folds sequentially to avoid memory blowup
    for train_idx, val_idx in kf.split(X_train):
        score = eval_one_fold(train_idx, val_idx, params, X_train, y_train_struct)
        if score is not None:
            fold_scores.append(score)

    if fold_scores:
        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            # write current best params to an outer file
            try:
                with open("best_param_rdm_forest.txt", "w") as fh:
                    json.dump(best_params, fh, default=lambda o: o.item() if hasattr(o, "item") else o, indent=2)
            except Exception:
                with open("best_param_rdm_forest.txt", "w") as fh:
                    fh.write(str(best_params))

    print(f"Iteration {i+1}: Mean score: {np.round(np.mean(fold_scores), 4) if fold_scores else 'nan'}", flush=True)
    gc.collect()

print("\nBest parameters :", best_params)
print("Best cross-validation score :", best_score)

# Fit final model with conservative resource usage
if best_params is None:
    raise RuntimeError("No valid hyperparameter configuration found during search.")

best_model_rf = RandomSurvivalForest(**best_params, random_state=0, n_jobs=1)
best_model_rf.fit(X_train, y_train_struct)

y_pred_rf = best_model_rf.predict(X_test)
ipcw_c_index_rf = concordance_index_ipcw(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=y_pred_rf
)

print(f"Random Forest IPCW Concordance Index on test : {ipcw_c_index_rf[0]}")

Random Search Progress:   0%|          | 0/200 [00:00<?, ?it/s]

Iteration 1: Mean score: 0.7027


Random Search Progress:   0%|          | 1/200 [00:12<43:03, 12.98s/it]

Iteration 2: Mean score: 0.7


Random Search Progress:   1%|          | 2/200 [02:35<4:55:07, 89.43s/it]

...

Iteration 68: Mean score: 0.6887


Random Search Progress:  34%|███▍      | 68/200 [1:55:52<2:11:17, 59.68s/it]

Fold failed with exception: censoring survival function is zero at one or more time points


Random Search Progress:  34%|███▍      | 68/200 [1:58:08<3:49:19, 104.24s/it]


KeyboardInterrupt: 

In [ ]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)

#We predict the survival function for the test set
y_pred_test = best_model_rf.predict(df_eval)

In [ ]:
#We export the predictions to a csv file

df = pd.read_csv("X_test\clinical_test.csv")

predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_Rdm_frst.csv', index=False)

## Deep Surv

In [ ]:
from utils.neural_network import DeepSurv
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
import matplotlib.pyplot as plt


model = DeepSurv(
    hidden_sizes = (64, 64, 64),   # three hidden layers of width 64
    dropout      = 0.3,
    l2_reg       = 1e-4,
    lr           = 1e-3,
    batch_size   = 128,
    epochs       = 300,
    patience     = 20,
    val_fraction = 0.15,
    random_state = 42,
    verbose      = 1,
)

model.fit(X_train, y_train)

: 